<h1><center> Graph RAG

## Qu'est-ce que le GraphRAG ?

Le GraphRAG (Graph Retrieval-Augmented Generation) est une évolution du RAG qui utilise une base de données orientée graphe (comme Neo4j) pour stocker les connaissances sous forme de Nœuds (entités : personnes, lieux, concepts) et de Relations (liens entre ces entités).

# 1. Creating graph documents

In [7]:
# chargement du fichier.pdf

from langchain_community.document_loaders import PyPDFLoader

loader =  PyPDFLoader("pdf_file_example.pdf")
documents = loader.load()


In [2]:
from langchain_ollama import ChatOllama
from langchain_experimental.graph_transformers import LLMGraphTransformer


llm = ChatOllama(model= "llama3.2", temperature= 0)
llm_transformer = LLMGraphTransformer(llm = llm)


graph_documents = llm_transformer.convert_to_graph_documents(documents)
print(graph_documents)

C:\Users\hp\AppData\Local\Temp\ipykernel_13200\3898502021.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


[GraphDocument(nodes=[Node(id='Le Semanticchunker', type='Concept', properties={}), Node(id='Découpage Sémantique', type='Term', properties={}), Node(id='Chunking', type='Concept', properties={}), Node(id='Méthode', type='Term', properties={})], relationships=[Relationship(source=Node(id='Le Semanticchunker', type='Concept', properties={}), target=Node(id='Découpage Sémantique', type='Term', properties={}), type='DESCRIBES', properties={}), Relationship(source=Node(id='Découpage Sémantique', type='Term', properties={}), target=Node(id='Chunking', type='Concept', properties={}), type='RELATED_TO', properties={})], source=Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-07-22T13:27:46+01:00', 'author': 'RAJAA LEBNAITI', 'moddate': '2026-07-22T13:27:46+01:00', 'source': 'pdf_file_example.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content="Le SemanticChunker (découpage sémantique) est l'une des méthodes de chun

### Results:
<pre>
[GraphDocument(nodes=

[Node(id='Le Semanticchunker', type='Concept', properties={}), 

Node(id='Découpage Sémantique', type='Term', properties={}), 

Node(id='Chunking', type='Concept', properties={}), 

Node(id='Méthode', type='Term', properties={})], 


relationships=

[Relationship(source=Node(id='Le Semanticchunker', type='Concept', properties={}), target=Node(id='Découpage Sémantique', type='Term', properties={}), type='DESCRIBES', properties={}), 

Relationship(source=Node(id='Découpage Sémantique', type='Term', properties={}), target=Node(id='Chunking', type='Concept', properties={}), type='RELATED_TO', properties={})], 

source=Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-07-22T13:27:46+01:00', 'author': 'RAJAA LEBNAITI', 'moddate': '2026-07-22T13:27:46+01:00', 'source': 'pdf_file_example.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content="Le SemanticChunker (découpage sémantique) est l'une des méthodes de chunking les plus \nintelligentes : au lieu de couper le texte aveuglément tous les X caractères ou tokens. il analyse le \nsens des phrases pour ne couper que là où l'idée change radicalement."))]

# 2. Storing graph documents using Neoj4 Database

In [18]:
# Connexion à la base de donnée Neoj4

from langchain_community.graphs import Neo4jGraph

graph = Neo4jGraph(
    url="neo4j+s://5f0bd404.databases.neo4j.io",
    username="5f0bd404",
    password="Tz9KM4a8ddrglvbVPZ4E2m2VXqQL0FKinQiijyWBQh0",
    database="5f0bd404"
)

In [19]:
# storing graph documents


# Initialisation du llm local via ollama
from langchain_experimental.graph_transformers import LLMGraphTransformer
llm = ChatOllama(model= "llama3.2", temperature= 0)


# convertir le texte vers graphe
llm_transformer = LLMGraphTransformer(llm = llm)



# extraction des noeuds et relations depuis le document
graph_documents = llm_transformer.convert_to_graph_documents(documents)


# Injection dans la base Neoj4
graph.add_graph_documents(
    graph_documents,
include_source=True,     # Lie le nœud Document d'origine aux entités créées
    baseEntityLabel=True      # Ajoute une étiquette générique `__Entity__` sur chaque nœud
)


print("Graphe importé avec succés: ")


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description="warn: feature deprecated with replacement. apoc.create.addLabels is deprecated. It is replaced by Cypher's dynamic labels; `SET n:$(labels)`..", position=<SummaryInputPosition line=1, column=257, offset=256>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 256, 'line': 1, 'column': 257}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MERGE (d:Document {id:$document.metadata.id}) SET d.text = $document.page_content SET d += $document.metadata WITH d UNWIND $data AS row MERGE (source:`__Entity__` {id: row.id}) SET source += row.properties MERGE (d)-[:MENTIONS]->(source) WITH source, row CALL apoc.create.addLabels( source, [row.type] 

Graphe importé avec succés: 


In [21]:
print(graph.get_schema)

Node properties:

Relationship properties:

The relationships:



# 3. Quering database using Cypher query language

In [23]:
result = graph.query("""
MATCH (s)-[r]->(o)
RETURN s.id AS sujet, type(r) AS relation, o.id AS objet
LIMIT 25
""")

for row in result:
    print(f"{row['sujet']} --[{row['relation']}]--> {row['objet']}")

87d30be7e7ff34f3e54a58502b343f46 --[MENTIONS]--> Project Alpha
87d30be7e7ff34f3e54a58502b343f46 --[MENTIONS]--> Techcorp
87d30be7e7ff34f3e54a58502b343f46 --[MENTIONS]--> Guido Van Rossum
87d30be7e7ff34f3e54a58502b343f46 --[MENTIONS]--> Neo4J Inc.
87d30be7e7ff34f3e54a58502b343f46 --[MENTIONS]--> San Francisco
87d30be7e7ff34f3e54a58502b343f46 --[MENTIONS]--> San Jose
Project Alpha --[LEAD_ARCHITECT]--> Guido Van Rossum
Project Alpha --[USES_DATABASE]--> Neo4J Inc.
Techcorp --[CREATED]--> Project Alpha
Guido Van Rossum --[WORKS_AT]--> Techcorp
Neo4J Inc. --[LOCATED_IN]--> San Jose


In [24]:
# Recherche les relations de supervision ou d'organisation
result = graph.query("""
MATCH (personne:Person)-[r]->(cible)
RETURN personne.id AS nom, type(r) AS role, cible.id AS lie_a
""")

for row in result:
    print(f"👤 {row['nom']} -> [{row['role']}] -> {row['lie_a']}")

👤 Guido Van Rossum -> [WORKS_AT] -> Techcorp
